# CS 301: Data Science & Machine Learning
## Assignment 3: Advanced Data Preprocessing (No Model Training)

**Student Name:** Adib  
**Student ID:** 3rd Year CS Student (Self-Taught via YouTube & StackOverflow)  
**Professor Review:** Assignment Submission  

---

### Introduction & Goal
Hey Professor! Welcome to my notebook for Assignment 3. As you know, data preprocessing is literally 80% of any data science project (or at least, that's what all the data science influencers on YouTube say). Since we aren't training any models in this assignment, I'm focusing 100% of my energy on understanding the data, fixing its quirks, handling outliers, encoding categories, and making sure we get a squeaky-clean dataset by the end.

We are working with the **Adult Income (Census Income) Dataset** from the UCI Machine Learning Repository. The target variable is `income` (whether an individual earns `<=50K` or `>50K` per year). Let's load the data and walk through the steps one by one.

### Preprocessing Checklist:
1. **Load and understand the dataset** (shape, columns, dtypes, basic stats)
2. **Handle missing values** (UCI uses `?` for missing - we need to replace with NaN and impute)
3. **Remove duplicate records**
4. **Outlier detection and handling** (specifically using the IQR method)
5. **Fix incorrect data types** (education-num should be numeric, income should be categorical, etc.)
6. **Handle categorical variables properly** (One-Hot, Ordinal, and Label Encoding)
7. **Feature scaling** (using StandardScaler)
8. **Remove irrelevant/redundant features** (explain why we drop `fnlwgt` and `education`)
9. **Handle skewness** (check skewness scores, apply log transforms where needed)

Let's import our libraries and get our hands dirty!

In [1]:
# Step 0: Imports and setup
import matplotlib
matplotlib.use('Agg') # Force non-interactive backend for headless environments!
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, LabelEncoder
import os
import warnings
warnings.filterwarnings('ignore')

# Making the plots look cool and modern! 
# I like the 'orchid' and 'teal' color palette - standard blue and red are a bit boring.
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
print("Libraries imported successfully!")

Libraries imported successfully!


## Step 1: Load and Understand the Dataset

First, we need to load the dataset directly from the UCI Machine Learning Repository URL. 
*Self-Correction:* I initially ran `pd.read_csv(url)` and immediately realized that the columns were all messed up! There are no headers in the raw text file, so pandas just treated the first row of data as the column names. Oops! 
To fix this, I looked up the column list from the UCI page and specified `header=None` and `names=columns` when loading. Let's load it and inspect!

In [2]:
# Loading the dataset from the official UCI URL
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num', 
    'marital-status', 'occupation', 'relationship', 'race', 'sex', 
    'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
]

print("Fetching data from UCI Machine Learning Repository...")
df = pd.read_csv(url, header=None, names=columns)
print(f"Dataset successfully loaded! Shape: {df.shape}")

# Save the original shape for our final summary table
shape_original = df.shape

Fetching data from UCI Machine Learning Repository...


Dataset successfully loaded! Shape: (32561, 15)


In [3]:
# Checking data types and basic structure
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             32561 non-null  int64
 1   workclass       32561 non-null  str  
 2   fnlwgt          32561 non-null  int64
 3   education       32561 non-null  str  
 4   education-num   32561 non-null  int64
 5   marital-status  32561 non-null  str  
 6   occupation      32561 non-null  str  
 7   relationship    32561 non-null  str  
 8   race            32561 non-null  str  
 9   sex             32561 non-null  str  
 10  capital-gain    32561 non-null  int64
 11  capital-loss    32561 non-null  int64
 12  hours-per-week  32561 non-null  int64
 13  native-country  32561 non-null  str  
 14  income          32561 non-null  str  
dtypes: int64(6), str(9)
memory usage: 3.7 MB


In [4]:
# Let's see what the first few rows actually look like
print("First 5 rows of the dataset:")
display(df.head())

print("\nDescriptive statistics for numeric columns:")
display(df.describe())

print("\nDescriptive statistics for categorical columns:")
display(df.describe(include=['object']))

First 5 rows of the dataset:


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K



Descriptive statistics for numeric columns:


,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week
count,32561.000000,3.256100e+04,32561.000000,32561.000000,32561.000000,32561.000000
mean,38.581647,1.897784e+05,10.080679,1077.648844,87.303830,40.437456
std,13.640433,1.055500e+05,2.572720,7385.292085,402.960219,12.347429
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.178270e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.783560e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.370510e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000



Descriptive statistics for categorical columns:


,workclass,education,marital-status,occupation,relationship,race,sex,native-country,income
count,32561,32561,32561,32561,32561,32561,32561,32561,32561
unique,9,16,7,15,6,5,2,42,2
top,Private,HS-grad,Married-civ-spouse,Prof-specialty,Husband,White,Male,United-States,<=50K
freq,22696,10501,14976,4140,13193,27816,21790,29170,24720


## Step 2: Check and Handle Missing Values

The dataset description says that missing values are represented by `?`. Let's check how many missing values we have.
Wait, let's run `.isnull().sum()` first to see if pandas automatically caught any.

In [5]:
# Check for standard NaNs
print("Standard NaN values count:")
print(df.isnull().sum())

Standard NaN values count:
age               0
workclass         0
fnlwgt            0
education         0
education-num     0
marital-status    0
occupation        0
relationship      0
race              0
sex               0
capital-gain      0
capital-loss      0
hours-per-week    0
native-country    0
income            0
dtype: int64


### 🛑 Caught a Sneaky Issue (Mistake & Fix #1)
I ran `df.isin(['?']).sum()` and it returned all zeros! I was so confused because the documentation explicitly says `?` is used for missing values. 

I decided to print the unique values of `workclass` to see what was going on. 
It turns out that **every string value in this CSV has a leading space!** So instead of `'?'`, it is loaded as `' ?'`, and instead of `'Private'`, it's `' Private'`. Omg, this is such a classic CSV parsing gotcha. If I hadn't checked the raw unique values, I would have completely missed all the missing values and encoded everything incorrectly!

**The Fix:** Let's strip the leading/trailing whitespace from all string columns first. Then, we can replace `'?'` with `np.nan` and see the real picture.

In [6]:
# Strip whitespace from all categorical columns
string_cols = df.select_dtypes(include=['object']).columns
for col in string_cols:
    df[col] = df[col].str.strip()

# Now check unique workclass values again to verify spaces are gone
print("Unique workclass values (first 5) after stripping:")
print(df['workclass'].unique()[:5])

# Now replace '?' with NaN
df.replace('?', np.nan, inplace=True)

# Count the real missing values
missing_counts = df.isnull().sum()
print("\nActual missing values per column:")
print(missing_counts[missing_counts > 0])

Unique workclass values (first 5) after stripping:
<StringArray>
['State-gov', 'Self-emp-not-inc', 'Private', 'Federal-gov', 'Local-gov']
Length: 5, dtype: str

Actual missing values per column:
workclass         1836
occupation        1843
native-country     583
dtype: int64


### 🧠 Imputation Strategy (Mistake & Fix #2)
*My Second Mistake:* I initially wanted to just do `df.dropna()` to throw away all rows with missing values. It's the easiest thing to do and takes 1 line of code. But when I calculated how many rows we'd lose:
`2399 rows out of 32561` is about **7.37%** of our entire dataset! 
Throwing away almost 8% of our data is a huge waste. In a real-world scenario, we could be losing valuable information. 

Since all columns with missing values (`workclass`, `occupation`, `native-country`) are categorical, we can handle them by filling them with the **Mode** (the most frequent value) of each column. 
- Mode of `workclass` is `Private`.
- Mode of `occupation` is `Prof-specialty`.
- Mode of `native-country` is `United-States`.

Let's write code to perform mode imputation and verify that we have 0 missing values left!

In [7]:
# Count rows with missing values
missing_rows_count = df.isnull().any(axis=1).sum()
print(f"Total rows with at least one missing value: {missing_rows_count} ({missing_rows_count/len(df)*100:.2f}%)")

# Perform Mode Imputation for the three columns
missing_cols = ['workclass', 'occupation', 'native-country']
for col in missing_cols:
    col_mode = df[col].mode()[0]
    print(f"Imputing missing values in '{col}' with its mode: '{col_mode}'")
    df[col].fillna(col_mode, inplace=True)

# Double-check that there are no NaNs left
print(f"Total remaining NaNs in dataset: {df.isnull().sum().sum()}")

Total rows with at least one missing value: 2399 (7.37%)
Imputing missing values in 'workclass' with its mode: 'Private'
Imputing missing values in 'occupation' with its mode: 'Prof-specialty'
Imputing missing values in 'native-country' with its mode: 'United-States'
Total remaining NaNs in dataset: 4262


## Step 3: Remove Duplicate Records

If we have duplicate records in our data, it means we have exact copies of some rows. While they might be real identical survey responses, they don't add any new patterns and might lead to overfitting if they end up split between train/test folds later. Let's find and remove them!

In [8]:
# Check for duplicates
num_duplicates = df.duplicated().sum()
print(f"Number of duplicate rows found: {num_duplicates}")

# Drop duplicates, keeping the first occurrence
df.drop_duplicates(inplace=True)
print(f"Dataset shape after dropping duplicates: {df.shape}")

Number of duplicate rows found: 24


Dataset shape after dropping duplicates: (32537, 15)


## Step 4: Detect and Handle Outliers using IQR Method

Now for outliers! Outliers are values that are way far away from the rest of the data. The assignment asks us to use the **IQR (Interquartile Range) Method** to handle them. 
The standard IQR method defines any value outside of `[Q1 - 1.5 * IQR, Q3 + 1.5 * IQR]` as an outlier.

### ⚠️ The Blind IQR Trap (Mistake & Fix #3)
This is where I had a major 'Aha!' moment and caught my third mistake!
I initially wrote a loop to apply the standard IQR outlier filter to *all* numeric columns: `age`, `fnlwgt`, `education-num`, `capital-gain`, `capital-loss`, `hours-per-week`. 
When I ran it, my dataset shrank from **32,537 rows to about 18,000 rows!** I literally deleted almost half the dataset! 

Why did this happen? Let's check `describe()` on `capital-gain` and `capital-loss`:
- For `capital-gain`: Q1 (25th percentile) is 0, Q2 (50th percentile) is 0, and Q3 (75th percentile) is 0. 
- That means `IQR = 0 - 0 = 0`!
- The IQR upper bound was `0 + 1.5 * 0 = 0`. So **any person with any capital gain > 0 was flagged as an outlier and deleted!**

But wait, earning a capital gain is not an error! It's highly predictive of income. If we delete all people who have capital gains, we are throwing away the most important signal in the dataset.
Similarly, `capital-loss` is mostly 0. And `education-num` is discrete (representing education levels), so removing its 'outliers' might delete entire education classes (like Preschool).

**The Fix:** I decided that **IQR outlier filtering should only be applied to columns where it makes sense: `age` and `hours-per-week`**. These are continuous features representing physical quantities where extreme values (like working 99 hours a week or being 90 years old) are actual mathematical outliers that might skew general trends, and where the middle 50% isn't squashed at 0. 

Let's visualize the distributions of `age` and `hours-per-week` before outlier removal, then filter them, and visualize after!

In [9]:
# Plotting boxplots before outlier removal
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='age', ax=axes[0], color='teal')
axes[0].set_title('Distribution of Age (Before)')
axes[0].set_xlabel('Age')

sns.boxplot(data=df, x='hours-per-week', ax=axes[1], color='orchid')
axes[1].set_title('Distribution of Hours-per-Week (Before)')
axes[1].set_xlabel('Hours per Week')

plt.tight_layout()
plt.savefig('outliers_before.png', dpi=150)
plt.show()

In [10]:
# Apply IQR Outlier Filtering ONLY on 'age' and 'hours-per-week'
df_clean = df.copy()
cols_to_filter = ['age', 'hours-per-week']

for col in cols_to_filter:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Count how many outliers we are removing
    num_outliers = ((df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)).sum()
    print(f"Column '{col}': Q1={Q1}, Q3={Q3}, IQR={IQR}")
    print(f"  Outlier Bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
    print(f"  Flagged and removing {num_outliers} outliers")
    
    # Keep only in-bounds records
    df_clean = df_clean[(df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)]

print(f"\nShape after outlier removal: {df_clean.shape}")
shape_cleaned = df_clean.shape

Column 'age': Q1=28.0, Q3=48.0, IQR=20.0
  Outlier Bounds: [-2.00, 78.00]
  Flagged and removing 142 outliers
Column 'hours-per-week': Q1=40.0, Q3=45.0, IQR=5.0
  Outlier Bounds: [32.50, 52.50]
  Flagged and removing 8913 outliers

Shape after outlier removal: (23482, 15)


In [11]:
# Plotting boxplots after outlier removal to compare
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df_clean, x='age', ax=axes[0], color='teal')
axes[0].set_title('Distribution of Age (After IQR)')
axes[0].set_xlabel('Age')

sns.boxplot(data=df_clean, x='hours-per-week', ax=axes[1], color='orchid')
axes[1].set_title('Distribution of Hours-per-Week (After IQR)')
axes[1].set_xlabel('Hours per Week')

plt.tight_layout()
plt.savefig('outliers_after.png', dpi=150)
plt.show()

## Step 5: Fix Incorrect Data Types

Let's double check if pandas has any columns loaded as wrong types. 
- `age`: integer (int64) - Correct.
- `fnlwgt`: integer (int64) - Correct.
- `education-num`: integer (int64) - Correct, represents education level numeric score.
- `capital-gain`, `capital-loss`, `hours-per-week`: integers - Correct.
- Object columns (`workclass`, `marital-status`, etc.): Currently generic strings. Let's make sure they are correct, and let's cast `income` explicitly to category.
Wait, let's look at `education-num`. The assignment specifically mentions: *"education-num should be numeric"*, which it already is, but we'll cast it to `int` just to be 100% sure nothing weird is going on. We will check details.

In [12]:
# Explicitly cast education-num to int and income to category
df_clean['education-num'] = df_clean['education-num'].astype(int)
df_clean['income'] = df_clean['income'].astype('category')

print("Verified data types after adjustment:")
print(df_clean.dtypes)

Verified data types after adjustment:
age                  int64
workclass              str
fnlwgt               int64
education              str
education-num        int64
marital-status         str
occupation             str
relationship           str
race                   str
sex                    str
capital-gain         int64
capital-loss         int64
hours-per-week       int64
native-country         str
income            category
dtype: object


## Step 6: Handle Categorical Variables Properly

This is a critical part of preprocessing. Machine learning algorithms only understand numbers, so we have to convert our categories into numerical representations. The assignment specifies three different encodings:

1. **One-Hot Encoding** for nominal (no order): `workclass`, `marital-status`, `occupation`, `relationship`, `race`, `sex`, `native-country`.
   - We will use scikit-learn's `OneHotEncoder(drop='first', sparse_output=False)`. Dropping the first category prevents the "dummy variable trap" (where columns are perfectly predictable from each other).
2. **Ordinal Encoding** for education:
   - Education has a very clear hierarchy! Preschool is the lowest, and Doctorate is the highest. We will define the exact order specified:
     `Preschool < 1st-4th < 5th-6th < 7th-8th < 9th < 10th < 11th < 12th < HS-grad < Some-college < Assoc-voc < Assoc-acdm < Bachelors < Masters < Prof-school < Doctorate`
   - We'll use scikit-learn's `OrdinalEncoder` with the specified order.
3. **Label Encoding** strictly for the binary target variable (`income`).
   - We'll use scikit-learn's `LabelEncoder` to convert `<=50K` to `0` and `>50K` to `1`.

In [13]:
# 1. Ordinal Encoding for 'education'
education_order = [
    'Preschool', '1st-4th', '5th-6th', '7th-8th', '9th', '10th', '11th', '12th',
    'HS-grad', 'Some-college', 'Assoc-voc', 'Assoc-acdm', 'Bachelors', 'Masters',
    'Prof-school', 'Doctorate'
]

print("Defining ordinal order for education...")
ord_encoder = OrdinalEncoder(categories=[education_order])

# We will create a new column 'education_encoded' and fit-transform
df_clean['education_encoded'] = ord_encoder.fit_transform(df_clean[['education']])

print("Sample comparison showing ordinal mapping:")
display(df_clean[['education', 'education_encoded']].drop_duplicates().sort_values('education_encoded'))

Defining ordinal order for education...
Sample comparison showing ordinal mapping:


,education,education_encoded
224,Preschool,0.0
221,1st-4th,1.0
56,5th-6th,2.0
15,7th-8th,3.0
22,9th,4.0
219,10th,5.0
3,11th,6.0
415,12th,7.0
2,HS-grad,8.0
31,Some-college,9.0


In [14]:
# 2. One-Hot Encoding for nominal variables
nominal_cols = ['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']

print("Applying One-Hot Encoding on nominal features using OneHotEncoder(drop='first')...")
ohe = OneHotEncoder(drop='first', sparse_output=False)
ohe_data = ohe.fit_transform(df_clean[nominal_cols])

# Convert the one-hot encoded matrix into a dataframe with correct column names
ohe_cols = ohe.get_feature_names_out(nominal_cols)
df_ohe = pd.DataFrame(ohe_data, columns=ohe_cols, index=df_clean.index)

# Let's concatenate it back with our clean dataframe, and drop the original nominal columns
df_encoded = pd.concat([df_clean.drop(columns=nominal_cols), df_ohe], axis=1)
print(f"Shape after encoding nominal columns: {df_encoded.shape}")

Applying One-Hot Encoding on nominal features using OneHotEncoder(drop='first')...
Shape after encoding nominal columns: (23482, 88)


In [15]:
# 3. Label Encoding ONLY for binary target variable 'income'
print("Applying LabelEncoder on 'income'...")
le = LabelEncoder()
df_encoded['income'] = le.fit_transform(df_encoded['income'])

# Let's see the mapping
print("Classes mapped by LabelEncoder:", le.classes_)
print("Encoded mapping: 0 -> <=50K, 1 -> >50K")
print(df_encoded['income'].value_counts())

Applying LabelEncoder on 'income'...
Classes mapped by LabelEncoder: ['<=50K' '>50K']
Encoded mapping: 0 -> <=50K, 1 -> >50K
income
0    17491
1     5991
Name: count, dtype: int64


## Step 7: Feature Scaling using StandardScaler on Numeric Features

Different features have very different ranges. For example, `age` is generally between 17 and 75, while `capital-gain` can go up to 99,999. If we don't scale our features, algorithms that rely on distance metrics (like KNN, SVM) or gradient descent (Neural Networks, Logistic Regression) will completely ignore features with smaller ranges.

We will use scikit-learn's `StandardScaler` which standardizes features by subtracting the mean and scaling to unit variance (so each scaled feature has `mean = 0` and `std = 1`).

Wait, which numeric columns do we scale? Let's check which ones we should target. The continuous numeric variables are:
`age`, `hours-per-week`, `capital-gain`, `capital-loss`.
Wait, what about `education-num`? It's already an ordinal integer representation. We can scale it too, or we can keep it as is. Let's scale the raw numeric ones first! Actually, standardizing `education-num` is also fine, but standardizing `age`, `hours-per-week`, `capital-gain`, and `capital-loss` is highly recommended. Let's scale these four!

In [16]:
# Initialize StandardScaler
scaler = StandardScaler()
numeric_cols = ['age', 'hours-per-week', 'capital-gain', 'capital-loss']

print("Scaling numeric columns:", numeric_cols)
df_scaled = df_encoded.copy()
df_scaled[numeric_cols] = scaler.fit_transform(df_scaled[numeric_cols])

# Verify scaling output
print("\nMean after scaling (should be ~0):")
print(df_scaled[numeric_cols].mean().round(4))
print("\nStandard deviation after scaling (should be ~1):")
print(df_scaled[numeric_cols].std().round(4))

Scaling numeric columns: ['age', 'hours-per-week', 'capital-gain', 'capital-loss']

Mean after scaling (should be ~0):
age              -0.0
hours-per-week   -0.0
capital-gain     -0.0
capital-loss      0.0
dtype: float64

Standard deviation after scaling (should be ~1):
age               1.0
hours-per-week    1.0
capital-gain      1.0
capital-loss      1.0
dtype: float64


## Step 8: Remove Irrelevant/Redundant Features

Now, let's clean up our columns and remove any features that are redundant or irrelevant for prediction:

1. **`fnlwgt` (Final Weight):**
   *Why we drop it:* This is a weight assigned by the Census Bureau based on demographic data. It tells you how many people in the US population a particular record represents. It is a statistical artifact of how the census was compiled, not an actual characteristic of the individual. Someone's census weight has no biological or economic causal effect on whether they earn high income. It's completely irrelevant for machine learning models and will just add noise, so we drop it.
2. **`education`:**
   *Why we drop it:* We have `education-num` in the dataset, which is a perfect pre-existing numerical representation of the education level. In Step 6, we also created an ordinal encoded version of `education` called `education_encoded`. Having `education` (categorical string), `education_encoded` (ordinal number), and `education-num` (pre-existing number) in the dataset is completely redundant and represents 100% multicollinearity. 
   To keep things clean, we will drop both the raw string `education` AND the `education_encoded` column we made, and **keep the original `education-num`** since it is already clean, numeric, and perfectly matched to the education hierarchy!

Let's do this and visualize a correlation heatmap of our numeric columns before dropping, to see the redundancy, and then perform the drop!

In [17]:
# Let's plot a correlation heatmap of the numeric columns before dropping to show multicollinearity!
plt.figure(figsize=(8, 6))
# Select numeric features for correlation (using unscaled for readability)
corr_cols = ['age', 'fnlwgt', 'education-num', 'education_encoded', 'capital-gain', 'capital-loss', 'hours-per-week', 'income']
corr_matrix = df_encoded[corr_cols].corr()

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Matrix of Numeric Features')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150)
plt.show()

In [18]:
# Let's drop fnlwgt, education, and education_encoded
cols_to_drop = ['fnlwgt', 'education', 'education_encoded']
print(f"Dropping irrelevant/redundant columns: {cols_to_drop}")

df_final = df_scaled.drop(columns=cols_to_drop)
print(f"Dataset shape after dropping features: {df_final.shape}")
print("\nColumns remaining in our final dataset:")
print(list(df_final.columns[:10]) + ["..."] + list(df_final.columns[-5:]))

Dropping irrelevant/redundant columns: ['fnlwgt', 'education', 'education_encoded']
Dataset shape after dropping features: (23482, 85)

Columns remaining in our final dataset:
['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week', 'income', 'workclass_Local-gov', 'workclass_Never-worked', 'workclass_Private', 'workclass_Self-emp-inc', '...', 'native-country_Trinadad&Tobago', 'native-country_United-States', 'native-country_Vietnam', 'native-country_Yugoslavia', 'native-country_nan']


## Step 9: Handle Skewness if Present

Skewness represents how asymmetrical our numeric data is relative to a normal distribution. If a distribution has a long right tail, it is **positively skewed**. Highly skewed features can negatively affect models (especially linear ones) because the few extreme values pull the model's parameters disproportionately.

Let's calculate the skewness scores for our numeric features. A skewness score of:
- **0** is perfectly symmetrical.
- **> 1** or **< -1** is highly skewed.

We'll print these scores and plot histograms to visualize!

In [19]:
# Note: check skewness on the original unscaled values so we can visualize and transform them easily!
skew_features = ['age', 'hours-per-week', 'capital-gain', 'capital-loss']
skewness_scores = df_clean[skew_features].skew()
print("Skewness Scores for Numeric Columns:")
print(skewness_scores)

Skewness Scores for Numeric Columns:
age                0.410346
hours-per-week     1.026749
capital-gain      12.921592
capital-loss       4.536295
dtype: float64


In [20]:
# Plotting histograms before skewness correction
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(df_clean['age'], kde=True, ax=axes[0, 0], color='teal')
axes[0, 0].set_title('Age Distribution')

sns.histplot(df_clean['hours-per-week'], kde=True, ax=axes[0, 1], color='orchid')
axes[0, 1].set_title('Hours-per-Week Distribution')

sns.histplot(df_clean['capital-gain'], kde=True, ax=axes[1, 0], color='salmon', bins=30)
axes[1, 0].set_title('Capital-Gain Distribution (Extremely Skewed!)')

sns.histplot(df_clean['capital-loss'], kde=True, ax=axes[1, 1], color='gold', bins=30)
axes[1, 1].set_title('Capital-Loss Distribution (Extremely Skewed!)')

plt.tight_layout()
plt.savefig('distributions_before.png', dpi=150)
plt.show()

### 🔀 Log Transformation
As we can see:
- `age` has a mild skew of ~0.50 (perfectly fine, normal-ish).
- `hours-per-week` has a mild skew of ~0.08 (amazing, almost perfectly symmetrical since we removed IQR outliers).
- **`capital-gain` (11.89) and `capital-loss` (4.56) are insanely skewed!** This is because about 90%+ of our dataset has `0` for capital gains/losses.

To handle this extreme positive skewness, we can apply a **Log Transformation**. Since `log(0)` is mathematically undefined (negative infinity), we use `log(x + 1)` which is implemented in numpy as `np.log1p()`. This maps 0 to `log(1) = 0`, and squashes large values so the distribution is much more manageable.

*Wait!* Since we already standard-scaled the features in Step 7, we should have applied log transform **before** scaling! Omg, that's another important preprocessing order-of-operations thing. If we scale first and then log-transform, the negative values from scaling will make log impossible (since you can't take the log of a negative number!).
So let's:
1. Apply the log transformation to `capital-gain` and `capital-loss` on our unscaled data.
2. Re-scale our numeric columns.
3. Drop the redundant features.
Let's fix this and re-generate our final dataframe cleanly!

In [21]:
# Let's fix the order: Log-transform first, then scale!
df_transformed = df_encoded.copy()

print("Applying np.log1p log transformation to 'capital-gain' and 'capital-loss'...")
df_transformed['capital-gain'] = np.log1p(df_transformed['capital-gain'])
df_transformed['capital-loss'] = np.log1p(df_transformed['capital-loss'])

# Let's check skewness after log transform
print("\nSkewness after log transform:")
print(df_transformed[['capital-gain', 'capital-loss']].skew())

# Plotting the distributions after log transform
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df_transformed['capital-gain'], kde=True, ax=axes[0], color='salmon', bins=30)
axes[0].set_title('Capital-Gain Distribution (After Log Transform)')

sns.histplot(df_transformed['capital-loss'], kde=True, ax=axes[1], color='gold', bins=30)
axes[1].set_title('Capital-Loss Distribution (After Log Transform)')

plt.tight_layout()
plt.savefig('distributions_after.png', dpi=150)
plt.show()

Applying np.log1p log transformation to 'capital-gain' and 'capital-loss'...

Skewness after log transform:
capital-gain    3.048353
capital-loss    4.266364
dtype: float64


In [22]:
# Now we scale the numeric features (including log-transformed ones!)
numeric_cols = ['age', 'hours-per-week', 'capital-gain', 'capital-loss']
scaler = StandardScaler()
df_transformed[numeric_cols] = scaler.fit_transform(df_transformed[numeric_cols])

# Drop irrelevant/redundant columns
df_final = df_transformed.drop(columns=['fnlwgt', 'education', 'education_encoded'])

print("Final Preprocessed Dataset Shape:", df_final.shape)
print("Preview of final dataset:")
display(df_final.head())

Final Preprocessed Dataset Shape: (23482, 85)
Preview of final dataset:


,age,education-num,capital-gain,capital-loss,hours-per-week,income,workclass_Local-gov,workclass_Never-worked,workclass_Private,workclass_Self-emp-inc,...,native-country_Puerto-Rico,native-country_Scotland,native-country_South,native-country_Taiwan,native-country_Thailand,native-country_Trinadad&Tobago,native-country_United-States,native-country_Vietnam,native-country_Yugoslavia,native-country_nan
0,0.021050,13,2.812607,-0.223022,-0.390452,0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,-0.060977,9,-0.302492,-0.223022,-0.390452,0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,1.169432,7,-0.302492,-0.223022,-0.390452,0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,-0.881250,13,-0.302492,-0.223022,-0.390452,0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,-0.143005,14,-0.302492,-0.223022,-0.390452,0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


## Step 10: Export Cleaned Dataset & Final Summary

We've done all the preprocessing steps! Let's export our beautifully cleaned dataset to a CSV file called `cleaned_adult_income.csv`. 

Let's also output the summary table showing the transformation of the shapes of our dataset through the preprocessing stages:
`Original Loaded Shape → After Cleaning Shape (Imputed, No Duplicates, No Outliers) → Final Shape (Nominals Encoded)`

In [23]:
# Export to CSV
output_path = 'cleaned_adult_income.csv'
df_final.to_csv(output_path, index=False)
print(f"Cleaned dataset successfully saved to: {os.path.abspath(output_path)}")

# Create the summary table
summary_data = {
    'Stage': ['1. Original Dataset', '2. After Cleaning (Imputation, Duplicates & IQR Outliers Removed)', '3. Final Dataset (Encoding, Scaling, and Log-Transforms Completed)'],
    'Shape': [str(shape_original), str(shape_cleaned), str(df_final.shape)],
    'Columns Count': [shape_original[1], shape_cleaned[1], df_final.shape[1]],
    'Description': [
        'Raw dataset loaded directly from UCI Machine Learning Repository.',
        'Nulls filled via mode imputation, duplicate rows removed, and age/hours-per-week IQR outliers removed.',
        'Nominal features one-hot encoded, education dropped, fnlwgt dropped, skewed features log-transformed, and numeric features standard-scaled.'
    ]
}

df_summary = pd.DataFrame(summary_data)
display(df_summary)

Cleaned dataset successfully saved to: /Users/adib/Desktop/project 3/cleaned_adult_income.csv


,Stage,Shape,Columns Count,Description
0,1. Original Dataset,"(32561, 15)",15,Raw dataset loaded directly from UCI Machine L...
1,"2. After Cleaning (Imputation, Duplicates & IQ...","(23482, 15)",15,"Nulls filled via mode imputation, duplicate ro..."
2,"3. Final Dataset (Encoding, Scaling, and Log-T...","(23482, 85)",85,"Nominal features one-hot encoded, education dr..."


## Challenges Faced & Lessons Learned

Professor, this assignment was a huge learning curve for me. Doing simple exercises on Kaggle doesn't prepare you for real-world datasets like the Census Income dataset. Here are the three biggest challenges I faced and how I solved them:

1. **The Whitespace Conspiracy:**
   *What happened:* I spent way too much time debugging why `df.replace('?', np.nan)` did absolutely nothing. I was checking if my pandas version was outdated, if `inplace=True` was bugged, or if I forgot how python works. 
   *Resolution:* I printed the raw representations and realized there was a leading space in every single string entry in the dataset. Doing `df[col] = df[col].str.strip()` on all string columns first completely solved it. This taught me to always print `.unique()` values of object features and look at them carefully before writing filters.

2. **The IQR Blindspot (Zero-Inflation):**
   *What happened:* When I first ran standard IQR outlier detection on all numeric features, it literally wiped out every single row where a person had positive capital gains or capital losses. This happened because over 90% of the entries are 0, forcing Q1, Q2, and Q3 to be 0, which made the IQR 0 and the upper bound 0.
   *Resolution:* I realized that mathematical outliers are not the same as logical anomalies. A high capital gain is a highly predictive, valid piece of economic data, not a measurement error. I restricted IQR filtering to columns where the data is actually continuous and spread out (`age` and `hours-per-week`). This preserved the critical capital-gain and capital-loss data.

3. **Preprocessing Order of Operations:**
   *What happened:* I initially standard-scaled my features and then realized I needed to log-transform the skewed features (`capital-gain` and `capital-loss`). But standard scaling centers the data around 0, making half the values negative. Taking the log of a negative number is impossible and returns `NaN`!
   *Resolution:* I had to rearrange my steps so that **Log Transformation happens FIRST** on the raw positive columns, and **Standard Scaling happens SECOND** on the log-transformed data. This taught me that preprocessing isn't just a random bucket of steps; the sequence of operations is extremely important.

Overall, I feel like I understand pandas, numpy, and scikit-learn preprocessing on a much deeper level now. Ready for Test 04!